In [1]:
# Import or install Sionna
import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"

import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import plotly.graph_objects as go

no_preview = False # Toggle to False to use the preview widget

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

2026-02-23 15:32:04.156574: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-23 15:32:04.203587: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-23 15:32:05.463772: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/data/hw/sionna_env/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py

In [ ]:
import os
import tensorflow as tf

2026-02-23 15:45:08.719886: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-23 15:45:08.765656: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-23 15:45:10.273171: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/data/hw/sionna_env/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py

In [2]:
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("LD_LIBRARY_PATH:", os.environ.get("LD_LIBRARY_PATH"))

CUDA_VISIBLE_DEVICES: 2,3
LD_LIBRARY_PATH: /opt/ros/humble/lib/x86_64-linux-gnu:/opt/ros/humble/lib::/usr/local/cuda/lib64::/usr/lib/x86_64-linux-gnu/gazebo-11/plugins:


In [3]:
print("TF:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("TF location:", tf.__file__)

TF: 2.20.0
Built with CUDA: True
GPUs: []
TF location: /data/hw/sionna_env/lib/python3.12/site-packages/tensorflow/__init__.py


W0000 00:00:1771829115.438548  307255 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices("GPU"))

TF version: 2.20.0
Built with CUDA: True
GPUs: []


In [2]:
# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
xml_path = "/data/hw/sionna/ws/scenes/kookmin/kookmin_fixed.xml"
scene_dir = os.path.dirname(xml_path)
temp_xml_path = os.path.join(scene_dir, "kookmin_tmp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(temp_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {temp_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(temp_xml_path, merge_shapes=False)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /data/hw/sionna/ws/scenes/kookmin/kookmin_tmp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__2                         | itu                 
elm__3                         | itu                 
elm__5                         | itu                 
elm__7                         | itu                 
elm__9                         | itu                 

[정의된 재질 목록]
 - 이름: itu             (Type: concrete, Thickness: [0.2])


In [3]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat_7",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

road_object_id = "elm__7"
if road_object_id in scene.objects:
    scene.objects[road_object_id].radio_material = red_road_mat
    print(f"[설정] 도로({road_object_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__7)를 빨간색으로 변경했습니다.


In [4]:
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__7' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 226개


In [5]:
x_vals = road_positions[:, 0]
z_vals = road_positions[:, 2] 
indices = list(range(len(road_positions)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_vals, y=z_vals,
    mode='markers',
    marker=dict(size=5, color='blue'),
    text=indices,
    hovertemplate='<b>Index: %{text}</b><br>X: %{x:.1f}<br>Z: %{y:.1f}<extra></extra>' # 라벨도 Z로 표기
))

fig.update_layout(
    title="도로 점 확인용 지도 (X - Z 평면)",
    xaxis_title="X Axis (East/West)",
    yaxis_title="y Axis (North/South)", # Y축 라벨을 Z축으로 변경
    width=1000, height=800,
    hovermode='closest'
)

fig.show()

In [6]:
# 1. 설정 및 초기화
target_pos = road_positions[0]
print("="*60 + f"\n[설정] 테스트 목표: {target_pos}\n" + "="*60)

# 기존 객체 제거
for name in ['Tx_1', 'Tx_2', 'Tx_3', 'Car_Marker']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

rx = Receiver(name="rx_car", position=target_pos)
rx.receive_antenna = scene.rx_array
scene.add(rx)
print(" -> 자동차(Rx) 배치 완료.")

cam = Camera(position=target_pos + np.array([0, 100, 100]), look_at=target_pos)
print("[시각화] 3D 뷰어 실행")
scene.preview(show_devices=True, point_picker=True)

[설정] 테스트 목표: [-190.90263    0.3     -138.76404]
 -> 자동차(Rx) 배치 완료.
[시각화] 3D 뷰어 실행


In [7]:
# 1. 설정 및 초기화
target_pos = road_positions[0]
print("="*60 + f"\n[설정] 테스트 목표: {target_pos}\n" + "="*60)

# 2. 장치 배치
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[-12.176, 25, 84.599]]#, [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1"]#, "Tx_2", "Tx_3"]

# 기존 객체 제거
for name in tx_names + ['Car_Marker']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at(target_pos)
    scene.add(tx)
    print(f" -> {tx_names[i]} 배치 완료.")

rx = Receiver(name="rx_car", position=target_pos)
rx.receive_antenna = scene.rx_array
scene.add(rx)
print(" -> 자동차(Rx) 배치 완료.")

# 3. 시뮬레이션
print("[연산] 경로 계산 시작...")
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 4. 결과 검증
a, tau = paths.cir()
if tf.size(a) > 0 and tf.reduce_sum(tf.abs(a)) > 0:
    print(f"\n✅ [성공] 전파 도달 (Amp Sum: {tf.reduce_sum(tf.abs(a)):.2e})")
else:
    print("\n❌ [실패] 전파 미도달 (장애물 또는 거리 문제)")

# 5. 시각화
cam = Camera(position=target_pos + np.array([0, 100, 100]), look_at=target_pos)
print("[시각화] 3D 뷰어 실행")
scene.preview(paths=paths, show_devices=True)

[설정] 테스트 목표: [-190.90263    0.3     -138.76404]
 -> Tx_1 배치 완료.
 -> 자동차(Rx) 배치 완료.
[연산] 경로 계산 시작...

❌ [실패] 전파 미도달 (장애물 또는 거리 문제)
[시각화] 3D 뷰어 실행


W0000 00:00:1771828258.452273  291273 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==============================================================================
# 0. 데이터 준비 & 이동 경로 계산
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

path_indices = [
    205, 203, 201, 199, 197, 195, 193, 190, 182, 183, 7, 5
]

waypoints = road_positions[path_indices].copy()
waypoints[:, 1] += 1.5 

# 거리 및 시간 계산 (60km/h)
diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)
cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]
speed_ms = 30.0 / 3.6
total_time = total_distance / speed_ms
delta_t = 0.5 

def get_pos_at_time(t):
    target_dist = np.clip(speed_ms * t, 0, total_distance)
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))
    
    seg_start = cumulative_dists[idx]
    seg_len = segment_dists[idx]
    if seg_len == 0: return waypoints[idx]
    
    ratio = (target_dist - seg_start) / seg_len
    pos = waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])
    return pos  # 이미 numpy array입니다.

time_steps = np.arange(0, total_time + delta_t, delta_t)

# ==============================================================================
# 1. 씬(Scene) 초기화
# ==============================================================================
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()): scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()): scene.remove(name)

scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

#tx_positions = [[-125.663, 56.367, -181.453], [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
#tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at([0,0,0]) 
    scene.add(tx)

start_pos = get_pos_at_time(0.0)
rx = Receiver(name="rx_car", position=start_pos)
rx.receive_antenna = scene.rx_array
scene.add(rx)

solver = PathSolver()

# ==============================================================================
# 2. 인터랙티브 위젯 (수정완료)
# ==============================================================================
output_widget = widgets.Output()

def update_simulation(frame_idx):
    t = time_steps[frame_idx]
    current_pos = get_pos_at_time(t) # 여기서 이미 numpy array로 받아옵니다.
    
    # 위치 업데이트
    rx.position = current_pos
    for name in tx_names:
        scene.transmitters[name].look_at(current_pos)
    
    # 경로 계산
    paths = solver(scene, max_depth=3, max_num_paths_per_src=10, samples_per_src=100000, 
                   diffuse_reflection=True, diffraction=True)
    
    with output_widget:
        output_widget.clear_output(wait=True)
        
        # 3D 뷰어
        scene.preview(paths=paths, show_devices=True, resolution=[800, 600])
        
        # 정보 출력
        a, _ = paths.cir()
        p_val = tf.reduce_sum(tf.abs(a)**2) if tf.size(a) > 0 else 0.0
        db_val = 10 * np.log10(p_val) if p_val > 0 else -np.inf
        
        # [수정] current_pos.numpy() -> current_pos (이미 numpy 배열이라 메서드 호출 불필요)
        print(f"⏱ Time: {t:.1f}s | 📍 Pos: {current_pos} | 📶 Power: {db_val:.2f} dB")

slider = widgets.IntSlider(
    value=0, min=0, max=len(time_steps)-1, step=1,
    description='Time Step:', layout=widgets.Layout(width='600px')
)

widgets.interactive_output(update_simulation, {'frame_idx': slider})

print("▼ 슬라이더를 움직여보세요.")
display(slider, output_widget)

▼ 슬라이더를 움직여보세요.


IntSlider(value=0, description='Time Step:', layout=Layout(width='600px'), max=71)

Output()

In [9]:
print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs:", tf.config.list_physical_devices("GPU"))

TF version: 2.20.0
Built with CUDA: True
GPUs: []


In [10]:
frame_idx = 0  # 보고 싶은 프레임 인덱스(원하는 값으로 바꿔)
t = time_steps[frame_idx]
current_pos = get_pos_at_time(t)

# 위치/방향 업데이트
rx.position = current_pos
for name in tx_names:
    scene.transmitters[name].look_at(current_pos)

# 경로 계산
paths = solver(scene, max_depth=3, samples_per_src=100000,
               diffuse_reflection=True, diffraction=True,synthetic_array=True)

# CIR 추출
a, tau = paths.cir(out_type="tf", normalize_delays=False)

print("t =", t)
print("a shape:", a.shape)
print("tau shape:", tau.shape)

InvalidArgumentError: GPU:0 unknown device.

In [ ]:
# 계산 결과 캐시: (frame_idx, tx_idx, rel_delay) -> (t, pos, df)
_pdp_cache = {}

out = widgets.Output()

tx_dropdown = widgets.Dropdown(
    options=[("Tx_1", 0)],#, ("Tx_2", 1), ("Tx_3", 2)],
    value=0,
    description="TX:",
    layout=widgets.Layout(width="200px")
)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(time_steps)-1,
    step=1,
    description="Time:",
    continuous_update=False,
    layout=widgets.Layout(width="600px")
)

# 0부터 시작하는 상대지연으로 볼지 옵션 (원하면 켜기)
rel_delay_chk = widgets.Checkbox(
    value=False,
    description="Relative delay (min τ = 0)",
    indent=False
)

def _compute_mapping_for_frame(frame_idx: int, tx_idx: int, rel_delay: bool):
    key = (frame_idx, tx_idx, rel_delay)
    if key in _pdp_cache:
        return _pdp_cache[key]

    t = float(time_steps[frame_idx])
    pos = get_pos_at_time(t)

    # 위치/방향 업데이트
    rx.position = pos
    for name in tx_names:
        scene.transmitters[name].look_at(pos)

    # 경로 계산
    paths = solver(
        scene,
        max_depth=3,
        samples_per_src=100000,
        diffuse_reflection=True,
        diffraction=True,
        synthetic_array=True
    )

    # CIR 추출 (절대 지연)
    a, tau = paths.cir(out_type="tf", normalize_delays=False)

    # 너 케이스 기준:
    # a: (1, 2, 3, 128, P, 1)
    # tau: (1, 3, P)
    tau_tx = tau[0, tx_idx, :]  

    # PDP: Rx편파(2) + Tx포트(128) 합산 -> (P,)
    pdp = tf.reduce_sum(tf.abs(a[0, :, tx_idx, :, :, 0])**2, axis=[0, 1])  # (P,)

    # ---------------------------
    # padding/가짜 경로 제거
    #   - tau가 음수(-1 같은)면 padding일 가능성이 큼
    # ---------------------------
    valid = tf.math.is_finite(tau_tx) & (tau_tx >= 0)
    idx_valid = tf.where(valid)[:, 0]  # (P_valid,)

    if tf.size(idx_valid) == 0:
        # 유효 경로 없음
        df = pd.DataFrame(columns=["path_idx", "tau_ns", "pdp_db"])
        _pdp_cache[key] = (t, pos, df)
        return _pdp_cache[key]

    tau_v = tf.boolean_mask(tau_tx, valid)   # (P_valid,)
    pdp_v = tf.boolean_mask(pdp, valid)      # (P_valid,)

    # 상대지연 옵션: min τ를 0으로 이동
    if rel_delay:
        tau_v = tau_v - tf.reduce_min(tau_v)

    # 지연 기준 정렬
    order = tf.argsort(tau_v)
    tau_s = tf.gather(tau_v, order).numpy() * 1e9  # ns
    pdp_s = tf.gather(pdp_v, order).numpy()
    pdp_db = 10*np.log10(pdp_s + 1e-30)

    # 정렬된 path_idx (원래 인덱스 기준)
    path_idx_sorted = tf.gather(idx_valid, order).numpy()


    df = pd.DataFrame({
        "path_idx": path_idx_sorted.astype(int),
        "tau_ns": tau_s,
        "pdp_db": pdp_db
        })

    _pdp_cache[key] = (t, pos, df)
    return _pdp_cache[key]

def _update_plot(_=None):
    frame_idx = frame_slider.value
    tx_idx = tx_dropdown.value
    rel_delay = rel_delay_chk.value

    with out:
        out.clear_output(wait=True)

        t, pos, df = _compute_mapping_for_frame(frame_idx, tx_idx, rel_delay)

        if df.empty:
            print(f"t={t:.2f}s | frame={frame_idx} | Tx_{tx_idx+1} : 유효 경로가 없습니다 (완전 차폐/패딩 제거 후 0개).")
            print(f"pos={pos}")
            return

        # 1) 매핑 테이블 출력 (path_idx ↔ 임펄스)
        display(df)

        # 2) CIR(PDP) stem + 라벨(path_idx)
        plt.figure(figsize=(8, 3.6))
        plt.stem(df["tau_ns"].values, df["pdp_db"].values, basefmt=" ")

        # 라벨: 각 임펄스 위에 path_idx
        for x, y, pid in zip(df["tau_ns"].values, df["pdp_db"].values, df["path_idx"].values):
            plt.text(x, y, str(pid), fontsize=9, ha="center", va="bottom")

        plt.xlabel("Delay τ (ns)")
        plt.ylabel("Power (dB)")
        plt.title(f"Tx_{tx_idx+1} PDP at t={t:.2f}s | frame={frame_idx}\npos={pos}")
        plt.grid(True)
        plt.show()

# 이벤트 연결
frame_slider.observe(_update_plot, names="value")
tx_dropdown.observe(_update_plot, names="value")
rel_delay_chk.observe(_update_plot, names="value")

display(widgets.VBox([widgets.HBox([frame_slider, tx_dropdown]), rel_delay_chk]), out)
_update_plot()

Output()

: 